# 02 流失归因：用 pandas 复现报告的五条结论

**目的**：不依赖 MySQL，用 pandas 把 `sql/` 里 15 条 SQL 的核心结论重算一遍，
确认报告中的每个数字都能被独立复现。

> 本 notebook 只依赖 `pandas`，执行结果已写入单元格。


In [1]:
import pandas as pd

pd.set_option("display.width", 120)

df = pd.read_csv("../data/telco.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["MonthlyCharges"] * df["tenure"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print(f"用户总数 {len(df):,}　流失 {df.Churn.sum():,}　流失率 {df.Churn.mean():.2%}")


用户总数 7,043　流失 1,869　流失率 26.54%


## 结论一：留存悬崖（Q04）

把在网时长分成 4 段，比较各段流失率。

In [2]:
bins = [-1, 6, 12, 24, 999]
labels = ["1. 0-6个月", "2. 6-12个月", "3. 1-2年", "4. 2年以上"]
df["lifecycle"] = pd.cut(df["tenure"], bins=bins, labels=labels)

stage = df.groupby("lifecycle", observed=True).agg(
    用户数=("Churn", "size"), 流失人数=("Churn", "sum"))
stage["流失率%"] = (stage["流失人数"] / stage["用户数"] * 100).round(2)
print(stage.to_string())
print(f"\n0-6 个月贡献了全部流失的 {stage.loc['1. 0-6个月','流失人数'] / df.Churn.sum():.1%}")


            用户数  流失人数   流失率%
lifecycle                   
1. 0-6个月   1481   784  52.94
2. 6-12个月   705   253  35.89
3. 1-2年    1024   294  28.71
4. 2年以上    3833   538  14.04

0-6 个月贡献了全部流失的 41.9%


## 结论二：高危组合（Q09）

合同 × 支付方式交叉，只保留样本量 > 100 的组合（对应 SQL 里的 `HAVING COUNT(*) > 100`）。

In [3]:
combo = df.groupby(["Contract", "PaymentMethod"]).agg(
    用户数=("Churn", "size"), 流失率=("Churn", "mean"))
combo = combo[combo["用户数"] > 100].sort_values("流失率", ascending=False)
combo["流失率"] = (combo["流失率"] * 100).round(2)
print(combo.head(5).to_string())

m = df[(df.Contract == "Month-to-month") & (df.PaymentMethod == "Electronic check")]
print(f"\n月付+电子支票：{len(m):,} 人，流失率 {m.Churn.mean():.2%}，"
      f"占全部流失人数 {m.Churn.sum() / df.Churn.sum():.1%}")


                                           用户数    流失率
Contract       PaymentMethod                         
Month-to-month Electronic check           1850  53.73
               Bank transfer (automatic)   589  34.13
               Credit card (automatic)     543  32.78
               Mailed check                893  31.58
One year       Electronic check            347  18.44

月付+电子支票：1,850 人，流失率 53.73%，占全部流失人数 53.2%


## 结论三：财务损失（Q11）

把流失折算成月度 MRR 损失，并对比流失/留存用户的客单价。

In [4]:
lost = df[df.Churn == 1]
keep = df[df.Churn == 0]
print(f"流失人数            {len(lost):,}")
print(f"每月 MRR 损失        {lost.MonthlyCharges.sum():,.2f} 元")
print(f"年化损失             {lost.MonthlyCharges.sum() * 12:,.0f} 元")
print(f"流失用户平均月租      {lost.MonthlyCharges.mean():.2f} 元")
print(f"留存用户平均月租      {keep.MonthlyCharges.mean():.2f} 元")
print(f"流失用户月租溢价      {(lost.MonthlyCharges.mean() / keep.MonthlyCharges.mean() - 1):.2%}")


流失人数            1,869
每月 MRR 损失        139,130.85 元
年化损失             1,669,570 元
流失用户平均月租      74.44 元
留存用户平均月租      61.27 元
流失用户月租溢价      21.51%


## 结论四：高价值流失名单（Q13 / Q06）

按合同类型分组，取流失用户中历史消费最高的 Top 3——等价于 SQL 的
`RANK() OVER(PARTITION BY Contract ORDER BY TotalCharges DESC)`。

In [5]:
churned = df[df.Churn == 1].copy()
churned["rank"] = churned.groupby("Contract")["TotalCharges"].rank(
    method="first", ascending=False)
top3 = churned[churned["rank"] <= 3].sort_values(["Contract", "rank"])
print(top3[["customerID", "Contract", "TotalCharges", "rank"]]
      .assign(TotalCharges=lambda d: d.TotalCharges.round(2))
      .to_string(index=False))


customerID       Contract  TotalCharges  rank
5287-QWLKY Month-to-month       7548.10   1.0
5502-RLUYV Month-to-month       7446.90   2.0
9090-SGQXL Month-to-month       7299.65   3.0
2889-FPWRM       One year       8684.80   1.0
0201-OAMXR       One year       8127.60   2.0
3886-CERTZ       One year       8109.80   3.0
5271-YNWVR       Two year       7856.00   1.0
9053-JZFKV       Two year       7752.30   2.0
1555-DJEQW       Two year       7723.90   3.0


## 结论五：营收集中度（Q12）

用 NTILE 的思路把用户按累计消费分成 10 等份，看**头部与长尾各贡献多少营收**。

> ⚠️ 口径陷阱：SQL 里 `累计消费金额 / 全盘总营收 * 100` 已经乘过 100，
> 所以 `0.53` 表示 **0.53%**，不是 53%。另外「Top 10%」按 NTILE 分桶是 705 人，
> 与“排名前 704 人”会有 0.04pp 的差异，属正常口径差。

In [12]:
import numpy as np

rev = df.sort_values("TotalCharges", ascending=False).reset_index(drop=True)
total = rev["TotalCharges"].sum()
n = len(rev)
rev["cum_share"] = rev["TotalCharges"].cumsum() / total
# 等价于 MySQL NTILE(10)：bucket = floor((排名-1) * 10 / n) + 1
rev["decile"] = ((np.arange(n) * 10) // n) + 1

print(f"总营收 {total:,.0f} 元，用户 {n:,} 人")
print(f"Top 10 名          占营收 {rev.cum_share.iloc[9]:.2%}（单人最高 {rev.TotalCharges.iloc[0] / total:.4%}）")
for pct in (0.01, 0.10, 0.20, 0.40):
    k = int(round(n * pct))
    print(f"Top {pct:>4.0%} 用户（{k:>4} 人）  占营收 {rev.cum_share.iloc[k - 1]:.2%}")

top1 = rev[rev.decile == 1]
rest = rev[rev.decile > 1]
print(f"\nNTILE 口径：Top 10%（{len(top1)} 人）占营收 {top1.TotalCharges.sum() / total:.2%}，"
      f"其余 90%（{len(rest)} 人）占 {rest.TotalCharges.sum() / total:.2%}")

need80 = int((rev.cum_share < 0.8).sum()) + 1
print(f"覆盖 80% 营收需要 {need80:,} 人（占全部用户的 {need80 / n:.1%}）")
print(f"头部单人 {rev.TotalCharges.iloc[0]:,.1f} 元，是均值 {rev.TotalCharges.mean():,.1f} 元的 "
      f"{rev.TotalCharges.iloc[0] / rev.TotalCharges.mean():.1f} 倍")


总营收 16,056,169 元，用户 7,043 人
Top 10 名          占营收 0.53%（单人最高 0.0541%）
Top   1% 用户（  70 人）  占营收 3.62%
Top  10% 用户（ 704 人）  占营收 30.79%
Top  20% 用户（1409 人）  占营收 53.66%
Top  40% 用户（2817 人）  占营收 81.53%

NTILE 口径：Top 10%（705 人）占营收 30.83%，其余 90%（6338 人）占 69.17%
覆盖 80% 营收需要 2,702 人（占全部用户的 38.4%）
头部单人 8,684.8 元，是均值 2,279.7 元的 3.8 倍


## 与报告的对照

| 结论 | 报告数字 | 本 notebook 复算 |
|---|---|---|
| 整体流失率 | 26.54% | 26.54% |
| 0–6 个月流失率 | 52.94% | 52.94% |
| 月付 + 电子支票 | 53.73%，占流失 53.2% | 一致 |
| 月度损失 | 139,130.85 元 | 一致 |
| 流失/留存月租 | 74.44 / 61.27（+21.5%） | 一致 |
| 营收集中度 | Top 20% → 53.7% | 一致（初版报告误写为 Top10 → 53%） |

下一步：`03_modeling.ipynb` 用 scikit-learn 做流失预测（需要 `pip install -r requirements.txt`）。
